# Review Intelligence: Multi-Agent E-Retail Analysis

## What You Will Learn

In this demo, you'll build a multi-agent system that analyzes customer reviews using Google ADK (Agent Development Kit) powered by Anthropic's Claude.

**Key concepts:**
- **Agent:** An AI assistant with specific instructions
- **output_key:** Where an agent saves its findings
- **ParallelAgent:** A team that works side by side (fan-out)
- **SequentialAgent:** An assembly line, one step after another
- **Fan-out/Fan-in:** Split work among specialists, then combine results

**System diagram:**
```
Customer Reviews
     |
     v
[Sentiment | Defects | Fit & Comfort | Delivery] <-- Fan-out (parallel)
     |
     v
Synthesizer <-- Fan-in
     |
     v
Fix-it Brief
```


## Step 0: Setup

We need two packages:
- **google-adk:** Google's Agent Development Kit
- **litellm:** A translator that lets ADK talk to Anthropic's Claude (ADK is built for Gemini)

Run the cell below to install them.


In [1]:
# Install packages
# This may take a minute
!pip install -q google-adk litellm


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Check API Key

The Anthropic API key should be in your environment as `ANTHROPIC_API_KEY`. LiteLlm reads it automatically.


In [2]:
# Check if the API key exists
import os

if os.environ.get('ANTHROPIC_API_KEY'):
    print('✓ API key found')
else:
    print('✗ API key missing - set ANTHROPIC_API_KEY environment variable')

✓ API key found


### Define the Model

We use **Claude Haiku 4.5**, the cheapest current Anthropic model. By defining it once, we can swap models in one place.


In [3]:
# Import LiteLlm from ADK (the translator between ADK and Anthropic)
from google.adk.models.lite_llm import LiteLlm

# Create the model once, reuse it for all agents
# Model string: anthropic/claude-haiku-4-5-20251001
MODEL = LiteLlm(model='anthropic/claude-haiku-4-5-20251001')

### Mock Review Data

Here are ~20 customer reviews for wireless earbuds sold on an Indian marketplace.


In [4]:
# A list is a collection of items in [ ] separated by commas
# Each review is a string (text in quotes)
reviews = [
    'Sound quality is amazing! Best earbuds under ₹3000.',
    'Battery dies in 2 hours. Very disappointed.',
    'Bluetooth pairing fails constantly. Waste of money.',
    'Comfortable fit, can wear for hours without pain.',
    'Arrived 5 days late. Box was damaged.',
    'Bass is great but highs are too sharp.',
    'Superb product. Worth every rupee!',
    'Left earbud stopped working after 1 week.',
    'Delivery was fast. Packaging bilkul perfect tha.',
    'Sound achha hai but battery bahut jaldi khatam ho jaati hai.',
    'Connection drops every 10 minutes. Unusable.',
    'Very light and comfortable. No ear fatigue.',
    'Received wrong color. Customer service not responding.',
    'Battery lasts 6+ hours. Impressed!',
    'Ear tips don\'t fit properly. Keep falling out.',
    'Excellent noise cancellation for the price.',
    'Took 10 days to deliver. No tracking update.',
    'Refund of ₹500 still pending after return.',
    'Pairing is seamless. Works perfectly with iPhone.',
    'Right earbud volume much lower than left. Defect?'
]

# Join the list into one text block
# \n\n means two line breaks (a blank line between reviews)
REVIEWS_TEXT = '\n\n'.join(reviews)

print(f'Loaded {len(reviews)} reviews')

Loaded 20 reviews


## Step 1: Your First Analyst

Let's create a single agent: a **Sentiment** analyst that summarizes overall sentiment and top themes.

**New concept: async/await**

Think of ordering at a restaurant:
- Without async: You freeze at the counter until the kitchen finishes your meal.
- With async: You place the order, the kitchen works while you wait at your table (you're not frozen).

In Jupyter, just write `await` in front of the call. That's all you need to remember.


In [5]:
# Import Agent class and the runner
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# Create the Sentiment analyst
# - name: What we call this agent
# - model: Which AI to use (the MODEL we defined above)
# - instruction: What the agent should do
# - output_key: Where to save the result (like a variable name)
sentiment_agent = Agent(
    name='sentiment',
    model=MODEL,
    instruction='Summarize the overall sentiment and top 3 themes from these reviews. Be concise.',
    output_key='sentiment'
)

# A helper function to run an agent and print its reply
# A function is a reusable recipe: give it inputs, it does work
async def ask(agent, text):
    runner = InMemoryRunner(agent)  # Create a runner for this agent
    # await means: wait for the agent to finish (async, like the restaurant)
    events = await runner.run_debug(text)
    # Loop through events and print model replies
    for e in events:
        if hasattr(e, 'content') and e.content:
            for part in e.content.parts:
                if hasattr(part, 'text'):
                    print(part.text)

# Run the agent on our reviews
await ask(sentiment_agent, REVIEWS_TEXT)

sentiment > # Review Summary

**Overall Sentiment:** Mixed - Positive sound quality undermined by significant reliability and service issues

**Top 3 Themes:**

1. **Sound Quality (Positive):** Amazing audio, great bass, excellent noise cancellation, and comfortable fit praised by multiple reviewers

2. **Battery & Connectivity (Negative):** Critical failures including short battery life (2 hours), constant Bluetooth pairing issues, frequent connection drops, and hardware defects (one earbud stopping)

3. **Delivery & Customer Service (Negative):** Delayed shipments (5-10 days), damaged/wrong packaging, unresponsive support, and pending refunds creating post-purchase frustration

**Verdict:** Product has strong audio capabilities but is severely hampered by poor quality control, battery performance, and inadequate customer support despite competitive pricing.
# Review Summary

**Overall Sentiment:** Mixed - Positive sound quality undermined by significant reliability and service issues

## Step 2: A Helper to Make Analysts

We need four analysts: Sentiment, Defects, Fit & Comfort, Delivery. Instead of copying the same code four times, we write a **function** (a recipe) that builds an analyst.

**What is a function?**
A function is a reusable block of code. You give it inputs (like ingredients), it does work, and gives you an output (like a dish). This avoids repeating yourself.


In [6]:
# A function that builds an analyst agent
# Inputs: name (e.g. 'defects'), focus (what to extract)
# Output: an Agent ready to use
def analyst(name, focus):
    # f-strings let you put variables inside text using {curly_braces}
    instruction = f'From the reviews, extract only {focus}. Be concise.'
    return Agent(
        name=name,
        model=MODEL,
        instruction=instruction,
        output_key=name  # Save result with the analyst's name
    )

# Create four analysts using our helper
sentiment_analyst = analyst('sentiment', 'overall sentiment and top themes')
defects_analyst = analyst('defects', 'product defects and quality issues')
comfort_analyst = analyst('comfort', 'fit, comfort, and wearability feedback')
delivery_analyst = analyst('delivery', 'delivery and packaging problems')

print('Created 4 analysts: sentiment, defects, comfort, delivery')

Created 4 analysts: sentiment, defects, comfort, delivery


## Step 3: Run Them in Parallel

Now we put all four analysts into a **ParallelAgent**. They work at the same time (fan-out), like four chefs each cooking one dish instead of one chef cooking four dishes in a row.

**Why parallel?**
Faster! Instead of waiting for each analyst one by one, they all work simultaneously.


In [7]:
# Import ParallelAgent
from google.adk.agents import ParallelAgent
import time

# Create a team of analysts that work in parallel
# sub_agents is a list [ ] of the four analysts
panel = ParallelAgent(
    name='panel',
    sub_agents=[sentiment_analyst, defects_analyst, comfort_analyst, delivery_analyst]
)

# Run the panel and time it
start = time.time()
runner = InMemoryRunner(panel)
events = await runner.run_debug(REVIEWS_TEXT)
elapsed = time.time() - start

# Print each analyst's output
for e in events:
    if hasattr(e, 'author') and hasattr(e, 'content') and e.content:
        for part in e.content.parts:
            if hasattr(part, 'text'):
                print(f'--- {e.author.upper()} ---')
                print(part.text)
                print()

print(f'Completed in {elapsed:.1f} seconds')

/var/folders/f5/nnd5ztdd39xd8dr0yqn8wbpm0000gn/T/ipykernel_89037/4038422572.py:7: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  panel = ParallelAgent(


delivery > # Delivery and Packaging Problems

1. **Arrived 5 days late. Box was damaged.**
2. **Took 10 days to deliver. No tracking update.**
3. **Received wrong color. Customer service not responding.**
4. **Refund of ₹500 still pending after return.**
comfort > # Fit, Comfort & Wearability Feedback

**Positive:**
- Comfortable fit, can wear for hours without pain
- Very light and comfortable. No ear fatigue
- Seamless pairing and connectivity (works perfectly with iPhone)

**Negative:**
- Ear tips don't fit properly. Keep falling out
- Battery dies quickly (2 hours, or drains fast)
- Bluetooth pairing fails constantly
- Connection drops every 10 minutes
defects > # Product Defects & Quality Issues

1. **Battery** - Dies in 2 hours
2. **Bluetooth** - Pairing fails constantly
3. **Audio Quality** - Highs are too sharp
4. **Hardware** - Left earbud stopped working after 1 week
5. **Packaging** - Box was damaged on arrival
6. **Bluetooth** - Connection drops every 10 minutes
7. **Ear Ti

## Step 4: Add the Synthesizer (Fan-in)

The four analysts saved their findings with `output_key` (sentiment, defects, comfort, delivery). Now we create a **Synthesizer** that reads all four outputs and combines them into one brief.

**Fan-in:** Combining results from multiple parallel workers into one final output.

**How the Synthesizer reads outputs:**
Use `{curly_braces}` in the instruction to inject saved values. `{sentiment}` pulls in the sentiment analyst's output.


In [8]:
# The Synthesizer agent
# Its instruction references {sentiment}, {defects}, {comfort}, {delivery}
# ADK automatically replaces these with the saved outputs
synthesizer = Agent(
    name='synthesizer',
    model=MODEL,
    instruction="""You are synthesizing findings from a panel of analysts.

Analyst outputs:
- Sentiment: {sentiment}
- Defects: {defects}
- Fit & Comfort: {comfort}
- Delivery: {delivery}

Write a one-page brief for the product team:
1. Top 3 problems ranked by severity
2. One recommended fix for each
3. One thing customers love

Be concise and actionable.""",
    output_key='brief'
)

print('Synthesizer ready')

Synthesizer ready


## Step 5: Run the Full System

Now we wrap the **ParallelAgent** (panel) and the **Synthesizer** into a **SequentialAgent**:
1. Panel runs first (fan-out: four analysts work in parallel)
2. Synthesizer runs second (fan-in: combines the four outputs)

This is the full Review Intelligence pipeline.


In [9]:
# Import SequentialAgent
from google.adk.agents import SequentialAgent

# Build the pipeline: panel first, then synthesizer
# sub_agents is a list [ ] in order
pipeline = SequentialAgent(
    name='review_intelligence',
    sub_agents=[panel, synthesizer]
)

# Run the full system
print('Running full Review Intelligence system...\n')
runner = InMemoryRunner(pipeline)
events = await runner.run_debug(REVIEWS_TEXT)

# Print outputs from each stage
for e in events:
    if hasattr(e, 'author') and hasattr(e, 'content') and e.content:
        for part in e.content.parts:
            if hasattr(part, 'text'):
                print(f'=== {e.author.upper()} ===')
                print(part.text)
                print()

/var/folders/f5/nnd5ztdd39xd8dr0yqn8wbpm0000gn/T/ipykernel_89037/2755084152.py:6: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  pipeline = SequentialAgent(


Running full Review Intelligence system...

sentiment > # Sentiment Analysis Summary

## Overall Sentiment
**Mixed to Negative** - Positive sound quality undermined by significant reliability and delivery issues.

## Top Themes

**Positive:**
- Sound quality (bass, clarity, noise cancellation)
- Comfort & fit
- Seamless pairing (when working)
- Battery longevity (6+ hours)
- Value for price

**Negative:**
- **Battery issues** (dies quickly, inconsistent performance)
- **Connectivity problems** (constant pairing failures, frequent drops)
- **Hardware defects** (earbud malfunction, imbalanced volume)
- **Delivery delays & packaging damage**
- **Poor customer service** (unresponsive, delayed refunds)
- **Quality control** (ear tips fit issues, durability concerns)

## Key Insight
Product has excellent audio specs but suffers from poor build quality, connectivity reliability, and post-sale support. Multiple unit failures suggest QC issues.
delivery > # Delivery and Packaging Problems

1. *

## Recap & Exercises

**What we built:**
1. Four specialist analysts (Sentiment, Defects, Comfort, Delivery) that run in parallel
2. A Synthesizer that combines their findings
3. A SequentialAgent pipeline: fan-out (parallel panel), then fan-in (synthesizer)

**Try these exercises:**

1. **Add a fifth analyst:** Create a "Value" analyst focused on "price, value for money, and refund issues". Add it to the panel and update the Synthesizer instruction to include `{value}`.

2. **Change the audience:** Modify the Synthesizer's instruction to write for the customer-support team instead of the product team. What should change?

3. **Hinglish-aware analyst:** Add a "Language" analyst that identifies reviews in Hinglish and extracts their sentiment separately. (Hint: "Hinglish reviews and their sentiment")

**Key takeaways:**
- Agents are AI assistants with instructions and output_key
- ParallelAgent = fan-out (work in parallel)
- SequentialAgent = assembly line (one after another)
- Use `{placeholders}` to read saved outputs
- async/await is just like waiting at a restaurant
